In [1]:
import os
import torch
from olmo.model import OLMo
import yaml
import torch.nn as nn
from collections import defaultdict
from typing import List, Tuple, Dict
import numpy as np
from olmo.data import build_train_dataloader
from olmo.config import TrainConfig
from tqdm import tqdm
from collections import OrderedDict
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torch.func import functional_call, jvp, vjp
from torch.optim import Adam, AdamW


# base_path = '/n/netscratch/kempner_sham_lab/Everyone/ameterez/random_basis/'
# run_name = '44081494_3'
# step = 'step1300-unsharded'

base_path = '/n/netscratch/sham_lab/Lab/pranavajitnair/continual_learning/'
run_name = '46273296_14'
step = 'step3749-unsharded'

experiment_path = os.path.join(base_path, run_name, step)
config_path = os.path.join(experiment_path, 'config.yaml')
optim_path = os.path.join(experiment_path, 'optim.pt')
model_path = os.path.join(experiment_path, 'model.pt')

In [2]:
def tree_map(fn, tree_a, tree_b):
    assert tree_a.keys() == tree_b.keys()
    return OrderedDict((k, fn(tree_a[k], tree_b[k])) for k in tree_a.keys())


def cross_entropy_loss_and_accuracy(logits, targets, loss_mask=None):
    B, T, V = logits.shape
    logits_flat = logits.view(B * T, V)
    targets_flat = targets.view(B * T)

    if loss_mask is not None:
        mask_flat = loss_mask.view(B * T)
        valid = mask_flat > 0
        logits_flat = logits_flat[valid]
        targets_flat = targets_flat[valid]

    loss = F.cross_entropy(logits_flat, targets_flat)

    with torch.no_grad():
        preds = logits_flat.argmax(dim=-1)
        acc = (preds == targets_flat).float().mean().item()

    return loss, acc


def train_step_gauss_newton_quadratic(
    model,
    params0,            # OrderedDict[name -> Tensor], fixed expansion point θ0
    batch,              # dict with 'input_ids'
    wd,                 # weight decay on (θ - θ0)
    optimizer,          # FrozenAdamW / AdamW on model.parameters()
    device,
    batch_size,
    micro_batch_size,   # micro-batch size along batch dim,
):
    """
    Gauss–Newton quadratic loss on the linearized model around params0,
    with manual gradient accumulation over micro-batches of size `micro_batch_size`.

      v(θ)   = J0 (θ - θ0)
      L_quad = L(f(θ0)) + <g0, v> + 1/2 <v, H_logit v>

    Here H_logit v is computed *analytically* for CE, so we avoid second-order autograd.
    """

    model.eval()  # determinism for Taylor expansion

    # ---- full batch inputs ----
    input_tokens_full = batch["input_ids"].to(device)  # [B, T]
    B = batch_size
    assert batch_size <= input_tokens_full.size(0)

    # ---- current θ (what Adam updates) ----
    curr_params = OrderedDict(
        (name, p.detach()) for name, p in model.named_parameters()
    )

    # Δθ = θ - θ0 (same for all micro-batches)
    dparams = tree_map(lambda p, p0: p - p0, curr_params, params0)

    optimizer.zero_grad()

    accum_grads = None
    total_loss_quad = 0.0
    total_loss_ce0 = 0.0
    total_acc = 0.0
    total_weight = 0.0  # sum of micro-batch sizes (normalized by B)

    # ---- loop over micro-batches ----
    for start in range(0, B, micro_batch_size):
        end = min(start + micro_batch_size, B)
        mb_input_tokens = input_tokens_full[start:end]   # [b, T]
        b = mb_input_tokens.size(0)
        weight = b / B
        total_weight += weight

        # build targets / masks for this micro-batch
        target_tokens = mb_input_tokens.clone()
        target_tokens[:, :-1] = mb_input_tokens[:, 1:]
        target_tokens[:, -1] = mb_input_tokens[:, -1]  # masked out
        loss_masks = torch.ones_like(target_tokens, dtype=torch.float32, device=device)
        loss_masks[:, 0] = 0.0

        def f(p):
            out = functional_call(
                model,
                p,
                (mb_input_tokens,),
                {}
            )
            return out.logits  # [b, T, V]

        logits0_raw, v_logits = jvp(f, (params0,), (dparams,))

        logits0 = logits0_raw.detach().requires_grad_(True)

        base_loss_ce, accuracy = cross_entropy_loss_and_accuracy(
            logits0, target_tokens, loss_masks
        )

        g0 = torch.autograd.grad(
            base_loss_ce,
            logits0,
            create_graph=False,
            retain_graph=False,
        )[0]

        with torch.no_grad():
            # softmax probs at θ0
            p = logits0_raw.softmax(dim=-1)          # [b, T, V]
            p_flat = p.view(-1, p.size(-1))         # [N, V]
            v_flat = v_logits.view_as(p_flat)       # [N, V]
            mask_flat = loss_masks.view(-1)         # [N]

            # weights per token (CE reduction='mean' over masked positions)
            Z = mask_flat.sum().clamp_min(1.0)      # avoid div by 0
            w_flat = mask_flat / Z                  # [N]

            # H v = w * (p ⊙ v - p * (p·v))
            pv = (p_flat * v_flat).sum(dim=-1, keepdim=True)        # [N, 1]
            Hv_flat = w_flat.unsqueeze(-1) * (p_flat * v_flat - p_flat * pv)
            Hv_logits = Hv_flat.view_as(v_logits)   # [b, T, V]

        # 5) ∇_θ L_quad(θ) = J0^T (g0 + H_logit v) for this micro-batch
        _, vjp_f = vjp(f, params0)
        # detach Hv_logits and g0 for safety; grads are only wrt params
        gn_cotangent = (g0 + Hv_logits).detach()
        (grad_params_gn_mb,) = vjp_f(gn_cotangent)  # OrderedDict[name -> grad]

        # weight grads by micro-batch size / global batch size
        if accum_grads is None:
            accum_grads = OrderedDict(
                (name, g * weight) for name, g in grad_params_gn_mb.items()
            )
        else:
            for name in accum_grads.keys():
                accum_grads[name] += grad_params_gn_mb[name] * weight

        # logging (also weighted)
        with torch.no_grad():
            quad_lin = (g0 * v_logits).sum()
            quad_quad = 0.5 * (v_logits * Hv_logits).sum()
            loss_quad_mb = base_loss_ce.detach() + quad_lin + quad_quad

            total_loss_quad += loss_quad_mb.item() * weight
            total_loss_ce0 += base_loss_ce.item() * weight
            total_acc += accuracy * weight

        # free some stuff ASAP
        del logits0, g0, Hv_logits, p, p_flat, v_flat, mask_flat, w_flat, pv, Hv_flat

    # normalize logs by total_weight (~1.0)
    if total_weight > 0:
        avg_loss_quad = total_loss_quad / total_weight
        avg_loss_ce0 = total_loss_ce0 / total_weight
        avg_acc = total_acc / total_weight
    else:
        avg_loss_quad = total_loss_quad
        avg_loss_ce0 = total_loss_ce0
        avg_acc = total_acc

    # ---- add weight decay and install grads into model.parameters() ----
    l2_term = 0.0
    for name, param in model.named_parameters():
        grad = accum_grads[name]
        if wd > 0.0:
            diff = param.detach() - params0[name]
            grad = grad + wd * diff
            l2_term = l2_term + (diff * diff).sum()
        param.grad = grad

    optimizer.step()

    if wd > 0.0:
        avg_loss_quad = avg_loss_quad + 0.5 * wd * l2_term.item()

    metrics = {
        "loss_quadratic": avg_loss_quad,
        "loss_ce_at_theta0": avg_loss_ce0,
        "accuracy": avg_acc,
    }

    return metrics


def train_step_gauss_newton_quadratic_bf16(
    model,
    params0_bf16,        # OrderedDict[name -> Tensor(bf16)], expansion point θ0
    batch,               # dict with 'input_ids'
    wd,                  # weight decay on (θ - θ0)
    optimizer,           # FrozenAdamW / AdamW on model.parameters()
    device,
    micro_batch_size,    # micro-batch size along batch dim
):
    """
    Gauss–Newton quadratic loss on the linearized model around params0_bf16,
    using bf16 for the linearized model (JVP/VJP and logits), fp32 for the
    main model + optimizer.

      v(θ)   = J0 (θ - θ0)
      L_quad = L(f(θ0)) + <g0, v> + 1/2 <v, H_logit v>

    H_logit v is computed analytically for cross-entropy.
    """

    model.eval()  # determinism for Taylor expansion etc.

    # ---- full batch inputs ----
    input_tokens_full = batch["input_ids"].to(device)  # [B, T]
    B = input_tokens_full.size(0)

    # ---- Δθ = θ - θ0 in bf16 (single extra param copy) ----
    dparams_bf16 = OrderedDict(
        (name, p.detach().to(torch.bfloat16) - params0_bf16[name])
        for name, p in model.named_parameters()
    )

    optimizer.zero_grad()

    accum_grads = None
    total_loss_quad = 0.0
    total_loss_ce0 = 0.0
    total_acc = 0.0
    total_weight = 0.0  # sum of micro-batch sizes / B

    # ---- loop over micro-batches ----
    for start in range(0, B, micro_batch_size):
        end = min(start + micro_batch_size, B)
        mb_input_tokens = input_tokens_full[start:end]   # [b, T]
        b = mb_input_tokens.size(0)
        weight = b / B
        total_weight += weight

        # build targets / masks for this micro-batch
        target_tokens = mb_input_tokens.clone()
        target_tokens[:, :-1] = mb_input_tokens[:, 1:]
        target_tokens[:, -1] = mb_input_tokens[:, -1]  # masked out
        loss_masks = torch.ones_like(
            target_tokens, dtype=torch.float32, device=device
        )
        loss_masks[:, 0] = 0.0

        # f: params -> logits for THIS micro-batch
        def f(p):
            # p: OrderedDict of bf16 params; model stays fp32, autocast handles casts
            with autocast(dtype=torch.bfloat16):
                out = functional_call(
                    model,
                    p,
                    (mb_input_tokens,),
                    {}
                )
            return out.logits  # [b, T, V] in bf16 under autocast

        # 1) logits0 = f(θ0), v = J0 Δθ for this micro-batch (both in bf16)
        with autocast(dtype=torch.bfloat16):
            logits0_raw_bf16, v_logits_bf16 = jvp(f, (params0_bf16,), (dparams_bf16,))

        # 2) cast logits0 to fp32 leaf for CE + g0
        logits0 = logits0_raw_bf16.to(torch.float32).detach().requires_grad_(True)

        base_loss_ce, accuracy = cross_entropy_loss_and_accuracy(
            logits0, target_tokens, loss_masks
        )

        # 3) g0 = ∂L/∂logits0 (fp32, first-order)
        g0 = torch.autograd.grad(
            base_loss_ce,
            logits0,
            create_graph=False,
            retain_graph=False,
        )[0]  # [b, T, V] fp32

        # 4) H_logit v: analytic Gauss–Newton HVP for CE (masked mean), in fp32
        with torch.no_grad():
            # softmax probs at θ0 (fp32)
            p = logits0_raw_bf16.to(torch.float32).softmax(dim=-1)  # [b, T, V]
            v_logits_fp32 = v_logits_bf16.to(torch.float32)         # [b, T, V]

            p_flat = p.view(-1, p.size(-1))            # [N, V]
            v_flat = v_logits_fp32.view_as(p_flat)     # [N, V]
            mask_flat = loss_masks.view(-1)            # [N]

            # weights per token (CE reduction='mean' over masked positions)
            Z = mask_flat.sum().clamp_min(1.0)         # avoid div by 0
            w_flat = mask_flat / Z                     # [N]

            # H v = w * (p ⊙ v - p * (p·v))
            pv = (p_flat * v_flat).sum(dim=-1, keepdim=True)        # [N, 1]
            Hv_flat = w_flat.unsqueeze(-1) * (p_flat * v_flat - p_flat * pv)
            Hv_logits_fp32 = Hv_flat.view_as(v_logits_fp32)         # [b, T, V]

        # 5) ∇_θ L_quad(θ) = J0^T (g0 + H_logit v), via VJP in bf16
        _, vjp_f = vjp(f, params0_bf16)  # vjp built at bf16 θ0

        gn_cotangent_fp32 = (g0 + Hv_logits_fp32).detach()          # [b, T, V] fp32
        gn_cotangent_bf16 = gn_cotangent_fp32.to(torch.bfloat16)    # match f's output dtype

        with autocast(dtype=torch.bfloat16):
            (grad_params_gn_mb_bf16,) = vjp_f(gn_cotangent_bf16)    # OrderedDict[name -> bf16]

        # cast grads to fp32 for accumulation / optimizer
        grad_params_gn_mb = OrderedDict(
            (name, g.to(torch.float32)) for name, g in grad_params_gn_mb_bf16.items()
        )

        # weight grads by micro-batch size / global batch size
        if accum_grads is None:
            accum_grads = OrderedDict(
                (name, g * weight) for name, g in grad_params_gn_mb.items()
            )
        else:
            for name in accum_grads.keys():
                accum_grads[name] += grad_params_gn_mb[name] * weight

        # logging (also weighted), all in fp32
        with torch.no_grad():
            quad_lin = (g0 * v_logits_fp32).sum()
            quad_quad = 0.5 * (v_logits_fp32 * Hv_logits_fp32).sum()
            loss_quad_mb = base_loss_ce.detach() + quad_lin + quad_quad

            total_loss_quad += loss_quad_mb.item() * weight
            total_loss_ce0 += base_loss_ce.item() * weight
            total_acc += accuracy * weight

        # free per-micro-batch heavy stuff
        del logits0, g0, p, p_flat, v_flat, mask_flat, w_flat, pv, Hv_flat
        del logits0_raw_bf16, v_logits_bf16, v_logits_fp32, Hv_logits_fp32

    # normalize logs by total_weight (~1.0)
    if total_weight > 0:
        avg_loss_quad = total_loss_quad / total_weight
        avg_loss_ce0 = total_loss_ce0 / total_weight
        avg_acc = total_acc / total_weight
    else:
        avg_loss_quad = total_loss_quad
        avg_loss_ce0 = total_loss_ce0
        avg_acc = total_acc

    # ---- add weight decay and install grads into model.parameters() ----
    l2_term = 0.0
    for name, param in model.named_parameters():
        grad = accum_grads[name]
        if wd > 0.0:
            # params0_bf16[name] -> fp32 to match param
            diff = param.detach() - params0_bf16[name].to(param.dtype)
            grad = grad + wd * diff
            l2_term = l2_term + (diff * diff).sum()
        param.grad = grad

    optimizer.step()

    if wd > 0.0:
        avg_loss_quad = avg_loss_quad + 0.5 * wd * l2_term.item()

    metrics = {
        "loss_quadratic": avg_loss_quad,
        "loss_ce_at_theta0": avg_loss_ce0,
        "accuracy": avg_acc,
    }

    return metrics

class FrozenAdamW(AdamW):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999),
                 eps=1e-8, weight_decay=0.0, amsgrad=False):
        super().__init__(params, lr=lr, betas=betas,
                         eps=eps, weight_decay=weight_decay,
                         amsgrad=amsgrad)
        self.freeze_v = False  # controls whether preconditioner v is frozen

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            lr = group["lr"]
            eps = group["eps"]
            wd = group["weight_decay"]
            amsgrad = group.get("amsgrad", False)

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad

                state = self.state[p]
                # State init
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)
                    state["exp_avg_sq"] = torch.zeros_like(p)
                    if amsgrad:
                        state["max_exp_avg_sq"] = torch.zeros_like(p)

                exp_avg = state["exp_avg"]
                exp_avg_sq = state["exp_avg_sq"]
                state["step"] += 1
                step_t = state["step"]

                # Decoupled weight decay (AdamW-style)
                if wd != 0.0:
                    p.mul_(1 - lr * wd)

                # First moment
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)

                # Second moment: either update or freeze
                if not self.freeze_v:
                    exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                    if amsgrad:
                        max_exp_avg_sq = state["max_exp_avg_sq"]
                        torch.maximum(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                        v_tensor = max_exp_avg_sq
                    else:
                        v_tensor = exp_avg_sq

                    bias_correction2 = 1.0 - beta2 ** step_t
                    v_hat = v_tensor / bias_correction2
                else:
                    # Preconditioner is frozen: treat exp_avg_sq (or max_exp_avg_sq) as v_hat directly
                    if amsgrad:
                        v_hat = state["max_exp_avg_sq"]
                    else:
                        v_hat = exp_avg_sq

                # Bias correction for m
                bias_correction1 = 1.0 - beta1 ** step_t
                m_hat = exp_avg / bias_correction1

                denom = v_hat.sqrt().add_(eps)
                p.addcdiv_(m_hat, denom, value=-lr)

        return loss

def set_lr(optimizer, lr):
    for g in optimizer.param_groups:
        g["lr"] = lr

In [3]:
with open(config_path, 'r') as file:
    config_data = yaml.safe_load(file)
optimizer_name = config_data['optimizer']['name']
print(optimizer_name)
os.environ['RANK'] = '1'
device = 'cuda:0'
# 1) Load model
model = OLMo.from_checkpoint(checkpoint_dir=experiment_path, device=device)

# 2) Expansion point θ0 (fixed copy of current params)
params0 = OrderedDict(
    (name, p.detach().to(device).clone())
    for name, p in model.named_parameters()
)

# 3) Load optimizer state dict from FSDP run
optim_state_dict = torch.load(optim_path, map_location=device)

# 4) Strip FSDP wrapper prefix from state keys
id_to_fqn: Dict[int, str] = {}
for group in optim_state_dict["param_groups"]:
    new_param_names = []
    for fqn, id in zip(group["param_names"], group["params"]):
        fqn = fqn.replace("module.", "")
        id_to_fqn[id] = fqn
        new_param_names.append(fqn)
    group["param_names"] = new_param_names
    group["params"] = new_param_names
for id in list(optim_state_dict["state"].keys()):
    optim_state_dict["state"][id_to_fqn[id]] = optim_state_dict["state"].pop(id)

# (param_groups in this state dict are FSDP-specific; we won't rely on them for mapping)

# 5) Extract hyperparams from the saved defaults / first param group
defaults = optim_state_dict.get("defaults", {})
pg0 = optim_state_dict["param_groups"][0]

lr = pg0.get("lr", defaults.get("lr", 1e-3))
betas = pg0.get("betas", defaults.get("betas", (0.9, 0.999)))
eps = pg0.get("eps", defaults.get("eps", 1e-8))
weight_decay = pg0.get("weight_decay", defaults.get("weight_decay", 0.0))
amsgrad = pg0.get("amsgrad", defaults.get("amsgrad", False))

# 6) Create a fresh FrozenAdamW with the current model params
optimizer = FrozenAdamW(
    model.parameters(),
    lr=lr,
    betas=betas,
    eps=eps,
    weight_decay=weight_decay,
    amsgrad=amsgrad,
)

# 7) Copy per-parameter AdamW state onto this optimizer, matching by *name*
name_to_param = {name: p for name, p in model.named_parameters()}
saved_state = optim_state_dict["state"]

for name, state in saved_state.items():
    if name not in name_to_param:
        # param no longer exists or was renamed; skip
        continue

    p = name_to_param[name]
    opt_state = optimizer.state[p]  # will create empty state if missing

    # step
    if "step" in state:
        opt_state["step"] = int(state["step"])

    # exp_avg
    if "exp_avg" in state:
        t = state["exp_avg"].to(device=device, dtype=p.dtype)
        if t.shape == p.shape:
            opt_state["exp_avg"] = t.clone()
        else:
            # shape mismatch -> ignore this old state for safety
            opt_state["exp_avg"] = torch.zeros_like(p)

    # exp_avg_sq
    if "exp_avg_sq" in state:
        t = state["exp_avg_sq"].to(device=device, dtype=p.dtype)
        if t.shape == p.shape:
            opt_state["exp_avg_sq"] = t.clone()
        else:
            opt_state["exp_avg_sq"] = torch.zeros_like(p)

    # max_exp_avg_sq (for amsgrad)
    if "max_exp_avg_sq" in state:
        t = state["max_exp_avg_sq"].to(device=device, dtype=p.dtype)
        if t.shape == p.shape:
            opt_state["max_exp_avg_sq"] = t.clone()
        else:
            opt_state["max_exp_avg_sq"] = torch.zeros_like(p)

# 8) Freeze the preconditioner from here on
optimizer.freeze_v = True

for name, p in model.named_parameters():
    st = optimizer.state.get(p, None)
    if st is None:
        print("No optimizer state for", name)
    else:
        print("Loaded state for", name, "step", st.get("step", 0))

# Now you can build your dataloader etc.
cfg = TrainConfig.load(config_path)
cfg.data.num_workers = 1
train_loader = build_train_dataloader(cfg)

adamw
Loaded state for transformer.wte.weight step 3749
Loaded state for transformer.ln_f.weight step 3749
Loaded state for transformer.blocks.0.k_norm.weight step 3749
Loaded state for transformer.blocks.0.q_norm.weight step 3749
Loaded state for transformer.blocks.0.attn_out.weight step 3749
Loaded state for transformer.blocks.0.ff_out.weight step 3749
Loaded state for transformer.blocks.0.att_proj.weight step 3749
Loaded state for transformer.blocks.0.ff_proj.weight step 3749
Loaded state for transformer.blocks.0.attn_norm.weight step 3749
Loaded state for transformer.blocks.0.ff_norm.weight step 3749
Loaded state for transformer.blocks.1.k_norm.weight step 3749
Loaded state for transformer.blocks.1.q_norm.weight step 3749
Loaded state for transformer.blocks.1.attn_out.weight step 3749
Loaded state for transformer.blocks.1.ff_out.weight step 3749
Loaded state for transformer.blocks.1.att_proj.weight step 3749
Loaded state for transformer.blocks.1.ff_proj.weight step 3749
Loaded stat

In [4]:
change = 1
B = config_data['global_train_batch_size']
lr = optimizer.param_groups[0]['lr']

for idx, batch in enumerate(train_loader):
    
    metrics = train_step_gauss_newton_quadratic(
        model=model,
        params0=params0,
        batch=batch,
        wd=0.0,
        optimizer=optimizer,
        device=device,
        batch_size = B,
        micro_batch_size=8
    )
    if idx == 0:
        if change == 0:
            set_lr(optimizer, lr * 2)
        elif change == 1:
            set_lr(optimizer, lr / 2)
            B = B//4
    print(metrics)

{'loss_quadratic': 3.27056060731411, 'loss_ce_at_theta0': 3.27056060731411, 'accuracy': 0.3868256639689207}
{'loss_quadratic': 3.2393806725740433, 'loss_ce_at_theta0': 3.238741934299469, 'accuracy': 0.39508492313325405}
{'loss_quadratic': 3.245461881160736, 'loss_ce_at_theta0': 3.2477086633443832, 'accuracy': 0.3877688180655241}
{'loss_quadratic': 3.215612515807152, 'loss_ce_at_theta0': 3.218037500977516, 'accuracy': 0.3934582695364952}
{'loss_quadratic': 3.30721715092659, 'loss_ce_at_theta0': 3.3084587901830673, 'accuracy': 0.3814073149114847}
{'loss_quadratic': 3.247975468635559, 'loss_ce_at_theta0': 3.2449773102998734, 'accuracy': 0.38908999413251877}
{'loss_quadratic': 3.220890447497368, 'loss_ce_at_theta0': 3.215129792690277, 'accuracy': 0.393076429143548}
{'loss_quadratic': 3.2451954036951065, 'loss_ce_at_theta0': 3.2363835871219635, 'accuracy': 0.38612689450383186}
{'loss_quadratic': 3.2827585637569427, 'loss_ce_at_theta0': 3.2712502032518387, 'accuracy': 0.38331652991473675}
{'

KeyboardInterrupt: 